# Proyecto 38. Informe visual reproducible de resultados precomputados

**Bloque:** Matplotlib (UT3) · **Tiempo orientativo:** 10–12 horas

Una organización te ha entregado resultados ya calculados de una evaluación con tres clases abstractas. Tu tarea no es entrenar ni mejorar un modelo: debes verificar que los datos son coherentes, crear un informe visual claro y comunicar con prudencia qué muestran y qué no muestran.

**Datos:**

- `../datos/matplotlib/38_curvas_evaluacion.csv`
- `../datos/matplotlib/38_matriz_confusion.csv`
- `../datos/matplotlib/38_metricas_clase.csv`
- [Ficha de datos](../datos/matplotlib/38_resultados_evaluacion.md)

**Entrega:** este cuaderno completado, la figura `salidas/38_informe_visual_resultados.png` y un texto alternativo final de 150–200 palabras.

## Objetivos y límites

Aplicarás Pandas para leer y revisar CSV, NumPy para formar y comprobar una matriz, y Matplotlib para diseñar una figura compuesta. Recuperarás también rutas relativas, funciones y exportación.

Los resultados son sintéticos y precomputados. No debes afirmar que el sistema funcionará igual con datos futuros, identificar causas de los desacuerdos ni proponer cambios de entrenamiento. Tu informe debe separar **resultado observado**, **límite** y **siguiente comprobación**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def ruta_datos(nombre):
    # Compatible al abrir el cuaderno desde la raíz o su carpeta.
    candidatas = [Path("datos") / "matplotlib" / nombre, Path("..") / "datos" / "matplotlib" / nombre]
    for candidata in candidatas:
        if candidata.exists():
            return candidata
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")


curvas = pd.read_csv(ruta_datos("38_curvas_evaluacion.csv"))
conteos_largos = pd.read_csv(ruta_datos("38_matriz_confusion.csv"))
metricas = pd.read_csv(ruta_datos("38_metricas_clase.csv"))

## Hito 1. Inspección y contrato de datos

Antes de visualizar:

1. muestra las tres tablas y sus tipos;
2. comprueba que no hay valores ausentes;
3. comprueba que `iteracion` está ordenada y vale de 1 a 8;
4. identifica las tres clases y el total de casos de la matriz;
5. escribe una frase que explique qué representa una fila de cada CSV.

Añade aserciones que detengan el cuaderno si falta una columna, hay nulos o no aparecen exactamente las clases `A`, `B` y `C`.

In [ ]:
# Completa aquí el hito 1.

# 1. Mostrar tablas y tipos de datos
for nombre, df in [("curvas", curvas), ("conteos_largos", conteos_largos), ("metricas", metricas)]:
    print(f"=== {nombre.upper()} ===")
    print(df.dtypes)
    display(df)

# 2. Aserciones de esquema y ausencia de nulos
assert {"iteracion", "referencia", "revision"}.issubset(curvas.columns), "Faltan columnas en curvas"
assert conteos_largos.shape[1] >= 3, "Faltan columnas en conteos_largos"
assert {"clase", "precision", "cobertura"}.issubset(metricas.columns), "Faltan columnas en metricas"

assert not curvas.isna().any().any(), "Hay valores ausentes en curvas"
assert not conteos_largos.isna().any().any(), "Hay valores ausentes en conteos_largos"
assert not metricas.isna().any().any(), "Hay valores ausentes en metricas"

# 3. Comprobar que 'iteracion' está ordenada y cubre exactamente de 1 a 8
assert list(curvas["iteracion"]) == list(range(1, 9)), "La columna iteracion no va ordenada de 1 a 8"

# 4. Identificar las tres clases exactas {'A', 'B', 'C'} y el total de casos
col_real, col_pred, col_conteo = conteos_largos.columns[:3]
clases_esperadas = {"A", "B", "C"}

assert set(metricas["clase"]) == clases_esperadas, "Clases inesperadas en metricas"
assert set(conteos_largos[col_real]) == clases_esperadas, "Clases reales inesperadas"
assert set(conteos_largos[col_pred]) == clases_esperadas, "Clases predichas inesperadas"

total_casos = int(conteos_largos[col_conteo].sum())
print(f"Clases verificadas: {sorted(clases_esperadas)} | Total de casos: {total_casos}")

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. Recorrer las tablas con `for` | **[U06.1]** — bucle `for` | La Unidad 05.4 presenta tuplas, pero no explica expresamente el desempaquetado `for nombre, df`. |
| 2. Mostrar tipos y tablas con `dtypes` y `display` | **[U18.6]** — inspección de datos, columnas y tipos | Ese apartado trata `dtypes` y muestra tablas con `display`. |
| 3. Comprobar columnas y estructura con `assert` | **[U18.6]** para `columns` y `shape`; **[U19.4–U19.5]** para control de calidad y comprobaciones | En U19.5 hay ejemplos con `assert`, pero `issubset` no se explica expresamente. |
| 4. Detectar valores ausentes con `isna()` | **[U19.4]**, subapartado «Ausencias y duplicados» | La teoría cubre `isna()` y el recuento con `sum()`. La cadena `.any().any()` no se explica expresamente. |
| 5. Comprobar el orden y el rango de iteraciones | **[U32.1]** para el orden de una serie; **[U05.4]** para `range`; **[U19.5]** para `assert` | La igualdad con `list(range(1, 9))` comprueba la secuencia exacta de 1 a 8, no solo que esté ordenada. |
| 6. Obtener nombres de columnas y definir clases permitidas | **[U18.6]** para `columns`; **[U05.2]** para cortes como `[:3]`; **[U08.4]** para conjuntos | El desempaquetado de `col_real, col_pred, col_conteo` tampoco está explicado explícitamente. |
| 7. Validar las clases observadas mediante conjuntos | **[U08.4]** para conjuntos; **[U19.4–U19.5]** para validar la calidad de los datos | La comprobación exacta con `set(...) == clases_esperadas` es una aplicación de esos conceptos, no un ejemplo explicado literalmente. |
| 8. Sumar los conteos y convertirlos a `int` | **[U19.4]** para `.sum()` en recuentos; **[U03.4]** para `int()` | La Unidad 06.2 también presenta la suma como patrón de procesamiento, aunque aquí se usa Pandas. |

En resumen, para el hito 1 usaría principalmente **U18.6 y U19.4–U19.5**, con **U06.1** para el bucle y referencias puntuales a **U05, U08 y U32** para los demás conceptos. En varios comentarios la referencia debe indicar que se trata de una **aplicación** de la teoría, porque detalles como `issubset`, `.any().any()` o el desempaquetado de tuplas no aparecen desarrollados de forma explícita.

## Hito 2. Reconstrucción y comprobación numérica

Construye una matriz NumPy de forma `(3, 3)` a partir de `conteos_largos`, usando el orden de clases `['A', 'B', 'C']`. La fila debe ser la clase real y la columna, la clase predicha.

A continuación calcula:

- el total de casos y la suma de la diagonal;
- la precisión por clase: diagonal / suma de cada columna;
- la cobertura por clase: diagonal / suma de cada fila.

Compara los arrays calculados con las columnas del CSV de métricas mediante `np.allclose`. Si no coinciden, no continúes con la visualización hasta explicar la discrepancia.

In [ ]:
# Completa aquí el hito 2.

orden_clases = ["A", "B", "C"]
col_real, col_pred, col_conteo = conteos_largos.columns[:3]

# Reconstruir matriz (3, 3): filas = clase real, columnas = clase predicha
tabla_pivote = (
    conteos_largos.pivot(index=col_real, columns=col_pred, values=col_conteo)
    .reindex(index=orden_clases, columns=orden_clases)
)
matriz_conf = tabla_pivote.to_numpy(dtype=int)
assert matriz_conf.shape == (3, 3), f"Forma inesperada: {matriz_conf.shape}"

# Cálculos sobre la matriz reconstruida
total = int(matriz_conf.sum())
aciertos_diagonal = int(np.trace(matriz_conf))

# Precisión: diagonal / suma por columna (axis=0)
precision_calc = np.diag(matriz_conf) / matriz_conf.sum(axis=0)
# Cobertura: diagonal / suma por fila (axis=1)
cobertura_calc = np.diag(matriz_conf) / matriz_conf.sum(axis=1)

# Alinear el CSV de métricas al orden ['A', 'B', 'C'] antes de comparar
metricas_ord = metricas.set_index("clase").reindex(orden_clases).reset_index()

assert np.allclose(precision_calc, metricas_ord["precision"].to_numpy()), "Discrepancia en precisión"
assert np.allclose(cobertura_calc, metricas_ord["cobertura"].to_numpy()), "Discrepancia en cobertura"

print(f"Total de casos: {total} | Suma de la diagonal: {aciertos_diagonal}")
print("Precisión verificada:", np.round(precision_calc, 4))
print("Cobertura verificada:", np.round(cobertura_calc, 4))

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. Definir el orden de clases y obtener los nombres de las columnas | **[U37.2]** — orientación de la matriz; **[U36.3]** — matrices de conteos y etiquetas; **[U18.6]** — nombres de columnas; **[U05.2]** — cortes de secuencias | Estas unidades respaldan las ideas por separado. El desempaquetado de `col_real, col_pred, col_conteo` no se explica expresamente. |
| 2. Crear la tabla con `pivot` y reordenarla con `reindex` | **[U37.2]** — significado y orientación de la matriz de confusión | `pivot` y `reindex` no se explican en los cuadernos 0–37; la referencia cubre el concepto de la matriz, no esos métodos de Pandas. |
| 3. Convertir la tabla a un array NumPy y comprobar su forma | **[U28.1]** — arrays y formas; **[U19.5]** — comprobaciones con `assert` | La conversión concreta mediante `to_numpy(dtype=int)` no aparece explicada expresamente. |
| 4. Calcular el total de casos y la suma de la diagonal | **[U37.2]** — total de casos, coincidencias en la diagonal y `np.trace` | La referencia corresponde directamente a la interpretación de la diagonal y al total. |
| 5. Calcular la precisión con la suma de cada columna | **[U37.3]** — significado de la precisión; **[U27.3]** — agregaciones por eje | La definición de precisión está en U37.3 y el uso de ejes para agregar, en U27.3. |
| 6. Calcular la cobertura con la suma de cada fila | **[U37.3]** — significado de la cobertura; **[U27.3]** — agregaciones por eje | La definición de cobertura está en U37.3 y el uso de ejes para agregar, en U27.3. |
| 7. Alinear las métricas con `set_index` y `reindex` | **[U19.2]** — selección por etiquetas e índices | Es una referencia parcial: `set_index` y `reindex` no se explican expresamente en los cuadernos 0–37. |
| 8. Comparar los arrays calculados con `np.allclose` | **[U29.4]** — comparación de valores decimales con `np.allclose` | Corresponde directamente a la comprobación de igualdad numérica con tolerancia. |
| 9. Mostrar los resultados con `print` y `np.round` | **[U29.4]** — precisión finita de los decimales | `np.round` no se explica expresamente; U29.4 sirve de contexto para la precisión decimal, no para el método concreto de redondeo. |

## Hito 3. Figura compuesta

Crea una sola figura con tres paneles y un título global:

1. **Curvas:** `referencia` y `revision` por iteración, con eje vertical de 0 a 1, leyenda y cuadrícula.
2. **Matriz:** `imshow` de la matriz reconstruida, filas reales, columnas predichas, escala explícita, barra de color y los nueve valores escritos en las celdas.
3. **Métricas:** barras agrupadas de precisión y cobertura por clase, escala de 0 a 1 y etiquetas numéricas.

Usa la interfaz `fig, ejes = plt.subplots(...)`, una disposición legible y `layout="constrained"`. Las categorías deben distinguirse mediante etiquetas y leyenda, no solo por color.

In [ ]:
# Completa aquí el hito 3.

fig, ejes = plt.subplots(1, 3, figsize=(15, 4.6), layout="constrained")

# Panel 1: Curvas de evolución por iteración
ejes[0].plot(curvas["iteracion"], curvas["referencia"], marker="o", linestyle="-", color="tab:blue", label="Referencia")
ejes[0].plot(curvas["iteracion"], curvas["revision"], marker="s", linestyle="--", color="tab:orange", label="Revisión")
ejes[0].set(title="Evolución por iteración", xlabel="Iteración", ylabel="Valor de la medida")
ejes[0].set_xticks(curvas["iteracion"])
ejes[0].set_ylim(0, 1)
ejes[0].grid(axis="y", alpha=0.25)
ejes[0].legend()

# Panel 2: Matriz de confusión etiquetada
limite_sup = int(matriz_conf.max()) + 2
im = ejes[1].imshow(matriz_conf, cmap="Blues", vmin=0, vmax=limite_sup)
fig.colorbar(im, ax=ejes[1], label="Número de casos")
ejes[1].set(title="Matriz de confusión", xlabel="Clase predicha", ylabel="Clase real")
ejes[1].set_xticks(np.arange(len(orden_clases)), orden_clases)
ejes[1].set_yticks(np.arange(len(orden_clases)), orden_clases)

umbral_contraste = limite_sup / 2
for fila in range(matriz_conf.shape[0]):
    for col in range(matriz_conf.shape[1]):
        valor = matriz_conf[fila, col]
        color_texto = "white" if valor > umbral_contraste else "black"
        ejes[1].text(col, fila, str(valor), ha="center", va="center", color=color_texto, fontweight="bold")

# Panel 3: Barras agrupadas de precisión y cobertura por clase
x = np.arange(len(orden_clases))
ancho = 0.36
barras_prec = ejes[2].bar(x - ancho / 2, metricas_ord["precision"], ancho, color="tab:blue", label="Precisión")
barras_cob = ejes[2].bar(x + ancho / 2, metricas_ord["cobertura"], ancho, color="tab:green", label="Cobertura")

ejes[2].set(title="Métricas por clase", xlabel="Clase", ylabel="Proporción")
ejes[2].set_xticks(x, orden_clases)
ejes[2].set_ylim(0, 1)
ejes[2].bar_label(barras_prec, fmt="%.2f", padding=2, fontsize=9)
ejes[2].bar_label(barras_cob, fmt="%.2f", padding=2, fontsize=9)
ejes[2].grid(axis="y", alpha=0.25)
ejes[2].legend()

fig.suptitle(f"Informe visual de resultados precomputados ({total} casos evaluados)", fontsize=13)
plt.show()

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. Crear la figura de tres paneles con `plt.subplots` y `layout="constrained"` | **[U35.1–U35.2]** — figuras compuestas y diseño | U35.1 explica la cuadrícula de ejes y U35.2 el ajuste del espacio con `layout="constrained"`. |
| 2. Dibujar y diferenciar las dos curvas | **[U32.2]** — comparación de series; **[U37.1]** — curvas con resultados precomputados | Las referencias cubren la comparación en un mismo eje y el contexto de curvas ya calculadas. |
| 3. Configurar etiquetas, marcas, límites, cuadrícula y leyenda de las curvas | **[U32.2]** — presentación y comparación de series; **[U37.1]** — escala de 0 a 1 | La escala completa de 0 a 1 está ejemplificada en U37.1; el conjunto de elementos visuales se aborda al comparar series en U32.2. |
| 4. Mostrar la matriz con `imshow`, escala explícita y barra de color | **[U36.1–U36.3]** — representación de matrices, escalas y matrices de conteos | U36.1 presenta `imshow` y la barra de color; U36.2 trata límites de escala; U36.3 aplica la representación a conteos. |
| 5. Etiquetar filas y columnas como clase real y clase predicha | **[U36.3]** — etiquetas de matrices; **[U37.2]** — orientación real/predicha | U37.2 especifica que las filas son reales y las columnas predichas. |
| 6. Escribir los conteos en las celdas y elegir el color del texto según el fondo | **[U36.3]** — anotaciones y contraste en matrices; **[U06.1]** — bucle `for` | U36.3 muestra cómo escribir valores en celdas; U06.1 respalda el recorrido de filas y columnas mediante bucles. |
| 7. Crear barras agrupadas desplazando cada métrica a ambos lados de la posición de la clase | **[U34.4]** — barras agrupadas; **[U37.3]** — métricas por clase | U34.4 cubre el tipo de gráfico y U37.3 el contexto de precisión y cobertura por clase. |
| 8. Configurar la escala, las etiquetas numéricas, la cuadrícula y la leyenda de las barras | **[U34.4]** — diseño de barras agrupadas; **[U37.3]** — métricas como proporciones de 0 a 1 | U37.3 respalda la escala de proporciones; el uso de barras agrupadas y leyenda se cubre en U34.4. |
| 9. Añadir el título global de la figura | **[U35.1–U35.2]** — composición y jerarquía visual | Esos apartados tratan los títulos globales y la organización de figuras compuestas. |

## Hito 4. Revisión, exportación y comunicación

Antes de exportar, revisa título, unidades o escalas, contraste y tamaño legible. Guarda la figura con ruta relativa en `salidas/38_informe_visual_resultados.png`, a 150 dpi y con `bbox_inches="tight"`.

Cierra con un texto alternativo de 150–200 palabras que contenga estas tres partes:

- **Resultado:** describe las curvas, los desacuerdos más visibles y la comparación de métricas.
- **Límite:** explica que los datos son sintéticos y precomputados; indica además una limitación de muestra o contexto.
- **Siguiente paso:** propone una comprobación concreta antes de decidir cambios.

No incluyas explicaciones causales ni promesas sobre resultados futuros.

In [ ]:
# Completa aquí el hito 4.
carpeta_salidas = Path("salidas")
carpeta_salidas.mkdir(exist_ok=True)

ruta_figura = carpeta_salidas / "38_informe_visual_resultados.png"
fig.savefig(ruta_figura, dpi=150, bbox_inches="tight")

assert ruta_figura.exists(), f"No se pudo exportar la figura en {ruta_figura}"
print("Figura exportada correctamente en:", ruta_figura)

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. Crear la carpeta `salidas` con una ruta relativa | **[U35.4]** — exportación reproducible y carpeta de salida | El apartado recomienda usar rutas relativas y crear previamente la carpeta. |
| 2. Guardar la figura como PNG con `savefig`, 150 dpi y `bbox_inches="tight"` | **[U35.4]** — formatos y opciones de exportación | El apartado muestra `savefig`, PNG, `dpi` y `bbox_inches="tight"`. |
| 3. Comprobar que el archivo existe mediante `assert` | **[U35.4]**, ejercicio 35.04 — exportación comprobada; **[U19.5]** — uso general de `assert` | El ejercicio 35.04 comprueba la existencia de la ruta exportada; U19.5 respalda la comprobación con aserciones. |

## Criterios orientativos de revisión

| Aspecto | Evidencia esperada |
| --- | --- |
| Reproducibilidad | Lectura por rutas relativas, comprobaciones y sin datos externos. |
| Corrección numérica | Matriz con orientación documentada y métricas verificadas con `np.allclose`. |
| Diseño visual | Tres paneles con títulos, escalas, unidades/etiquetas, leyendas y barra de color donde corresponde. |
| Accesibilidad | Contraste suficiente, valores escritos en la matriz y explicación textual. |
| Comunicación responsable | Resultado, límite y siguiente paso sin atribuir causalidad ni extrapolar. |

**Ampliación voluntaria:** añade una segunda figura que normalice cada fila de la matriz como proporción, pero conserva también la matriz de conteos y explica por qué ambas responden a preguntas distintas.

## BLOQUE 2 · Complemento Matplotlib (Unidades 31–37 fuera del Proyecto 38)

En esta sección se agrupan todos los conceptos, herramientas y tipos de gráficos evaluables en el módulo que **no aparecen en el Proyecto 38**.

### 1. Dispersión y relación entre variables (`scatter` — Unidad 33)
- **Uso:** Evaluar la relación visual entre dos variables numéricas diferenciando grupos mediante marcadores y colores independientes.
- **Regla:** Una asociación visual o coeficiente de correlación (`df["x"].corr(df["y"])`) muestra relación en la muestra observada, **nunca causalidad**.

```python
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
estilos = {"A": ("o", "tab:blue"), "B": ("s", "tab:orange")}

for grupo, subdf in df.groupby("zona"):
    marcador, color = estilos[grupo]
    ax.scatter(subdf["x"], subdf["y"], marker=marcador, color=color, alpha=0.85, label=f"Zona {grupo}")

ax.set(title="Relación X vs. Y por zona", xlabel="Variable X (u.)", ylabel="Variable Y (u.)")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()
```

---

### 2. Histogramas comparables (`hist` — Unidad 34)
- **Uso:** Mostrar la frecuencia (conteo de observaciones) dentro de intervalos (*bins*), nunca valores individuales.
- **Regla:** Para comparar dos grupos de forma justa hay que compartir exactamente los mismos límites de intervalo (`bins`). Una media por sí sola oculta la dispersión, la asimetría y los valores atípicos.

```python
limites = np.arange(10, 22, 2)  # Intervalos idénticos: [10, 12, 14, 16, 18, 20]

fig, ejes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True, layout="constrained")
ejes[0].hist(grupo_a, bins=limites, color="tab:blue", edgecolor="white")
ejes[0].set(title="Grupo A", xlabel="Tiempo (min)", ylabel="Frecuencia (casos)")

ejes[1].hist(grupo_b, bins=limites, color="tab:orange", edgecolor="white")
ejes[1].set(title="Grupo B", xlabel="Tiempo (min)")

fig.suptitle("Comparación de distribuciones con intervalos idénticos")
plt.show()
```

---

### 3. Barras horizontales ordenadas (`barh` — Unidad 34)
- **Uso:** Preferible a `bar` cuando las etiquetas de las categorías son largas o numerosas, evitando rotaciones incomodas de texto.
- **Regla:** Ordenar las categorías previamente por magnitud con `np.argsort()` facilita la lectura limpia de menor a mayor.

```python
orden = np.argsort(valores)  # Orden ascendente por magnitud

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
barras = ax.barh(np.array(categorias)[orden], np.array(valores)[orden], color="tab:teal")
ax.set(title="Magnitud por categoría", xlabel="Cantidad (u.)", ylabel="Categoría")
ax.bar_label(barras, padding=3)
ax.grid(axis="x", alpha=0.25)
plt.show()
```

---

### 4. Anotaciones puntuales sin causalidad (`annotate` — Unidad 35)
- **Uso:** Señalar un valor máximo, mínimo o punto destacado en una coordenada concreta mediante texto y flecha.
- **Regla:** La anotación debe describir exclusivamente **qué valor y en qué posición** ocurre; nunca debe atribuir causas no demostradas dentro del gráfico.

```python
idx_max = valores.argmax()

fig, ax = plt.subplots(figsize=(6.5, 4), layout="constrained")
ax.plot(dias, valores, marker="o")
ax.annotate(
    f"Máximo observado: {valores[idx_max]} (día {dias[idx_max]})",
    xy=(dias[idx_max], valores[idx_max]),                  # Coordenada del punto
    xytext=(dias[idx_max] - 1.5, valores[idx_max] + 1.0),  # Posición del texto
    arrowprops={"arrowstyle": "->"},
)
ax.set(title="Serie con punto destacado", xlabel="Día", ylabel="Valor (u.)")
ax.set_xticks(dias)
ax.grid(axis="y", alpha=0.25)
plt.show()
```

---

### 5. Imágenes RGB (3 canales) y orientación `origin` (`imshow` — Unidad 36)
- **Orientación `origin="lower"`:** Coloca la fila `0` en la parte inferior del eje vertical (coordenadas cartesianas) en lugar de arriba (`origin="upper"`, valor por defecto en matrices de datos).
- **Comparación de matrices:** Exige fijar los mismos límites de escala (`vmin` y `vmax`) y el mismo mapa de color (`cmap`) en todos los paneles.
- **Imágenes RGB `(alto, ancho, 3)`:** Combinan tres canales independientes (rojo, verde y azul) por píxel. **No llevan `cmap` ni `colorbar`** porque no representan una única magnitud escalar ordenada.

```python
imagen_rgb = np.zeros((6, 6, 3), dtype=float)
imagen_rgb[:3, :, 0] = 0.85   # Canal 0: Rojo
imagen_rgb[3:, :, 2] = 0.85   # Canal 2: Azul
imagen_rgb[2:4, 2:4, 1] = 0.6  # Canal 1: Verde

fig, ax = plt.subplots(figsize=(4, 4), layout="constrained")
ax.imshow(imagen_rgb, interpolation="nearest")
ax.set(title="Imagen RGB (6×6×3)", xlabel="Columna de píxel", ylabel="Fila de píxel")
ax.set_xticks(np.arange(6))
ax.set_yticks(np.arange(6))
plt.show()
```